# 2 · Yeni versiyon (v2) — düzeltilmiş boru hattı

Aynı fikir, altı düzeltme, **2026 verisiyle**.

Bu notebook **kendi kendine yeter** — `01`'i çalıştırmana gerek yok, hiçbir yerden
import etmiyor. Baştan sona okuyunca tüm mantık burada.

## Neyi düzelttim

| # | Sorun (rapor kodu) | Düzeltme |
|---|---|---|
| 1 | **B1** — 10 üzerinden puan hiç 10'a çıkmıyor (turizmde tavan 5) | Taban kütlesi 1. banda, kalan 9 eşit dilime. Tavan gerçekten 10 |
| 2 | **B2** — hiçbir şey değişmese bile puanlar oynuyor | İndeks artık *eğitilmiş model*: ölçekleyici + PCA + sınırlar diske yazılıyor |
| 3 | **B9** — PCA işareti keyfî, sessizce ters dönebilir | Yük toplamı negatifse bileşen çevriliyor. Deterministik |
| 4 | **B5** — halka kenarı ağırlığı: 10 m'deki kafe = 499 m'deki kafe | Her POI'nin gerçek metresi kullanılıyor |
| 5 | **B6** — "en yakın metro 39 km" gibi anlamsız değerler | Kategori başına hizmet yarıçapı; ötesi `NULL` |
| 6 | **B11** — macOS yolları, `name` sütunu olmayan dosyalarda çökme | Tek ayar hücresi, güvenli varsayılanlar |

**Bonus:** 2026 verisiyle gelen kamera + yanğından mühafizə verisinden **yedinci indeks**
(`təhlükəsizlik`).

> **Dürüst not:** 4 numaralı düzeltme sıralamayı neredeyse hiç değiştirmiyor
> (Spearman 0.994–0.999). Savunulabilirlik ve hız kazancı, doğruluk kazancı değil.
> Asıl fark yaratanlar 1, 2 ve 3.

> **Çalıştırma:** yukarıdan aşağıya. ~2 dakika (eski + yeni veri, iki tam geçiş).

## Ayarlar

In [ ]:
# ==========================================================================
# AYARLAR
#   Yollar config/paths.yml dosyasindan okunur.
# ==========================================================================
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import h3
from sklearn.neighbors import BallTree
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from urbangeoai import config  # noqa: E402

settings = config.load()

NEW    = settings.paths.raw_current      # guncel veri surumu
OLD    = settings.paths.raw_previous     # onceki surum, karsilastirma icin
OUT    = settings.paths.output / "revised"
CACHE  = settings.paths.cache
REFDIR = settings.paths.reference
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

R_EARTH = 6371000.0
METERS_PER_MINUTE = 80.0
CUTOFF_M = 3000

print("NEW var mi:", NEW.exists(), "| OLD var mi:", OLD.exists() if OLD else None)

## Veri yükleme

Eski ve yeni veri **farklı dosya adları ve farklı sütun adları** kullanıyor.
Tek bir yükleyici ikisini de normalize ediyor.

In [ ]:
def _read(base, fname, enc="utf-8-sig"):
    """Bir POI dosyasını oku ve sütunları standartlaştır."""
    d = pd.read_csv(os.path.join(base, fname), encoding=enc, engine="python")
    # -> Sütun adları dosyadan dosyaya değişiyor: 'long'/'lon', 'Lat'/'lat', BOM'lu başlıklar...
    d.columns = [c.strip().lower() for c in d.columns]
    d = d.rename(columns={"long": "lon", "longitude": "lon", "latitude": "lat"})
    d = d.loc[:, ~d.columns.duplicated()]
    d["lat"] = pd.to_numeric(d["lat"], errors="coerce")
    d["lon"] = pd.to_numeric(d["lon"], errors="coerce")
    # -> DÜZELTME 6 (B11): dayanacaq.csv ve Ts Kamera.csv'de 'name' sütunu YOK.
    # -> Orijinal kod bunlarda KeyError veriyordu. Dosya adından bir varsayılan üretiyoruz.
    if "name" not in d.columns:
        d["name"] = fname.rsplit(".", 1)[0]
    return d.dropna(subset=["lat", "lon"]).reset_index(drop=True)

def _t(d, val):
    """Tek dosyadaki çok kategoriyi ayır (ör. qida.csv -> kafe / restoran / ...)."""
    return d[d["type"].astype(str).str.strip().str.lower() == val.lower()].reset_index(drop=True)

def metro_gdf():
    """Metro OSM'den gelir. cache/metro.csv yanında hazır; yoksa Overpass'tan çeker."""
    cache = str(CACHE / "metro.csv")
    if os.path.exists(cache):
        d = pd.read_csv(cache)
    else:
        import requests
        q = ('[out:json];node["railway"="station"]["station"="subway"]'
             '(40.2,49.5,40.6,50.4);out body;')
        # -> Overpass sık 504 veriyor; üç aynayı sırayla dene
        data = None
        for url in ["https://overpass-api.de/api/interpreter",
                    "https://overpass.kumi.systems/api/interpreter",
                    "https://overpass.osm.ch/api/interpreter"]:
            try:
                r = requests.get(url, params={"data": q},
                                 headers={"User-Agent": "geoinvest/1.0"}, timeout=90)
                r.raise_for_status(); data = r.json(); break
            except Exception:
                continue
        if data is None:
            raise RuntimeError("Overpass cevap vermedi; cache/metro.csv koy.")
        d = pd.DataFrame([{"id": e["id"], "lat": e["lat"], "lon": e["lon"],
                           "name": e.get("tags", {}).get("name", "Unnamed")}
                          for e in data["elements"] if e["type"] == "node"]).drop_duplicates(subset="name")
        # -> Depo/servis noktaları istasyon değil. 'Məhəmməd Hadi' sondaki boşluk hatası da burada düzeldi.
        d = d[~d["name"].str.contains(r"Azər Neft Yağ|Məhəmməd Hadi|B-4|Y14|Y15|Depo|depos",
                                      case=False, na=False)].reset_index(drop=True)
        d.to_csv(cache, index=False)
    return d

def load_pois(which):
    """Kategori adı -> koordinat tablosu. Anahtarlar iki veri setinde de aynı."""
    if which == "old":
        school, qida = _read(OLD, "school_data.csv"), _read(OLD, "qida.csv")
        return {
            "metro": metro_gdf(),                      "kdgr": _t(school, "uşaq bağçası"),
            "schl": _t(school, "məktəb"),              "univ": _read(OLD, "ali_mekteb.csv"),
            "bus": _read(OLD, "dayanacaq.csv"),        "shopping": _read(OLD, "shopping_malls.csv"),
            "parks": _read(OLD, "parks.csv"),          "museum": _read(OLD, "muzey.csv"),
            "theatre": _read(OLD, "theatre.csv"),      "coffee": _t(qida, "kafe"),
            "restaurant": _t(qida, "restoran"),        "fast_food": _t(qida, "fast-food"),
            "bars": _t(qida, "bar"),                   "sport": _read(OLD, "sport_facility.csv"),
            # -> eski hotel.csv UTF-8 değil; ayrıca bu iki dosya BÜTÜN Azərbaycanı kapsıyor
            "hotel": _read(OLD, "hotel.csv", "ISO-8859-1"),
            "monuments": _read(OLD, "historic_monuments.csv"),
            "concert_hall": _read(OLD, "concert_halls.csv"),
        }
    school, qida = _read(NEW, "mekteb,bagca.csv"), _read(NEW, "qida.csv")
    return {
        "metro": metro_gdf(),                          "kdgr": _t(school, "uşaq bağçası"),
        "schl": _t(school, "məktəb"),                  "univ": _read(NEW, "Ali mekteb.csv"),
        "bus": _read(NEW, "dayanacaq.csv"),            "shopping": _read(NEW, "Ticaret merkezi.csv"),
        "parks": _read(NEW, "Park.csv"),               "museum": _read(NEW, "muzey.csv"),
        "theatre": _read(NEW, "Teatr.csv"),            "coffee": _t(qida, "kafe"),
        "restaurant": _t(qida, "restoran"),            "fast_food": _t(qida, "fast-food"),
        "bars": _t(qida, "bar"),                       "sport": _read(NEW, "idman zali.csv"),
        "hotel": _read(NEW, "hotel.csv"),              "monuments": _read(NEW, "Tarixi abide.csv"),
        "concert_hall": _read(NEW, "konsert zali.csv"),
        # -> 2026'da GELEN YENİ KATEGORİLER — 2023 sisteminde karşılığı yok
        "camera": _read(NEW, "Ts Kamera.csv"),         "fire": _read(NEW, "Yangindan muhafize.csv"),
    }

p_new = load_pois("new")
print(f"{sum(len(v) for k,v in p_new.items() if k!='metro'):,} POI, {len(p_new)} kategori")
for k, v in p_new.items(): print(f"  {k:<13}{len(v):5d}")

## Izgara

`01` ile aynı: sabit kutu → H3 res-10 → kara maskesiyle kırp.

In [ ]:
BBOX = [[49.5, 40.2], [50.4, 40.2], [50.4, 40.6], [49.5, 40.6], [49.5, 40.2]]

def grid(resolution=10):
    poly = h3.LatLngPoly([[lat, lon] for lon, lat in BBOX])
    idx = h3.polygon_to_cells(poly, resolution)
    lat, lon = zip(*[h3.cell_to_latlng(i) for i in idx])
    df = pd.DataFrame({"h3_index": list(idx), "latitude": lat, "longitude": lon})
    # -> Kara maskesi. Orijinal map_absheron.geojson kayıp (S1); GADM Abşeron ikamesi.
    g = gpd.read_file(str(CACHE / "gadm41_AZE_1.json"))
    mask = g[g["NAME_1"].str.contains("Baku|Absheron", case=False)]
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
    out = gpd.clip(gdf, mask.to_crs(gdf.crs))
    return pd.DataFrame(out[["h3_index", "latitude", "longitude"]]).reset_index(drop=True)

cells = grid(10)
print(f"{len(cells):,} H3 res-10 kara hücresi")

## Düzeltme 4 · Sürekli mesafe azalımı

**Orijinal:** POI'leri 5 halkaya böl, her halkadaki *herkese* halkanın dış kenar
ağırlığını ver. 10 m'deki kafe ile 499 m'deki kafe aynı puan.

**Yeni:** her POI'nin **gerçek metresini** kullan.

`query_radius` 3 km içindeki tüm POI'leri mesafeleriyle döndürüyor; her birine
kendi mesafesine göre ağırlık veriliyor.

In [ ]:
# -> β: mesafe arttıkça ağırlık ne kadar hızlı düşsün. Orijinaldeki değerler korundu,
# -> iki yeni kategori eklendi. Bunların nereden geldiği hâlâ belgesiz — S6.
DECAY = {
    "kdgr": 0.1, "schl": 0.2, "univ": 0.1, "bus": 0.7, "metro": 0.05,
    "shopping": 0.1, "parks": 0.1, "museum": 0.1, "theatre": 0.1,
    "coffee": 0.5, "restaurant": 0.3, "fast_food": 0.3, "bars": 0.2,
    "sport": 0.2, "hotel": 0.1, "monuments": 0.1, "concert_hall": 0.05,
    "camera": 0.3, "fire": 0.05,     # -> yeni: kamera yakın olmalı, itfaiye uzaktan da hizmet verir
}

def continuous_score(cells, poi, beta, cutoff=CUTOFF_M):
    """Σ  exp(-β × o POI'ye yürüme dakikası)  —  cutoff içindeki HER POI için."""
    tree = BallTree(np.radians(poi[["lat", "lon"]].values), metric="haversine")
    cc = np.radians(cells[["latitude", "longitude"]].values)
    # -> return_distance=True: sadece sayma, mesafeleri de getir. Farkın tamamı burada.
    idx, dist = tree.query_radius(cc, r=cutoff / R_EARTH, return_distance=True)
    out = np.empty(len(cells), dtype=float)
    k = beta / METERS_PER_MINUTE          # -> metreyi doğrudan ağırlığa çeviren katsayı
    for i, d in enumerate(dist):
        out[i] = np.exp(-k * (d * R_EARTH)).sum() if d.size else 0.0
    return out

# -> Farkı göster: aynı kategoride iki yöntem yan yana
demo = cells.head(3000)
tree = BallTree(np.radians(p_new["coffee"][["lat","lon"]].values), metric="haversine")
band = np.zeros(len(demo)); prev = 0
for b in [500,1000,1500,2000,3000]:
    cur = tree.query_radius(np.radians(demo[["latitude","longitude"]].values), r=b/R_EARTH, count_only=True)
    band += (cur - prev) * np.exp(-DECAY["coffee"] * b / METERS_PER_MINUTE); prev = cur
cont = continuous_score(demo, p_new["coffee"], DECAY["coffee"])
print(f"halka yöntemi ortalama : {band.mean():.4f}")
print(f"sürekli yöntem ortalama: {cont.mean():.4f}   (halka yöntemi olduğundan düşük gösteriyor)")

## Düzeltme 3 · PCA işaretini sabitle

`sklearn`'ün birinci bileşeninin işareti keyfîdir. Yükler negatif çıkarsa indeks
tersine döner — en erişilebilir yer 1, en kötü yer 10 olur, **hiçbir hata mesajı vermeden**.

Orijinal kod bunu sadece ekrana yazdırıyor, kontrol etmiyor. İki satırla sabitliyoruz.

In [ ]:
def pca_index(X_df):
    """PC1, işareti 'yüksek = daha erişilebilir' olacak şekilde sabitlenmiş."""
    X = StandardScaler().fit_transform(X_df)   # -> her kategoriyi aynı teraziye getir
    p = PCA(n_components=1)
    comp = p.fit_transform(X).ravel()
    load = p.components_[0]
    # -> ASIL DÜZELTME: yüklerin çoğunluğu negatifse her şeyi ters çevir
    if load.sum() < 0:
        comp, load = -comp, -load
    return comp, load * np.sqrt(p.explained_variance_[0]), p.explained_variance_ratio_[0]

## Düzeltme 1 · 1–10 skalası gerçekten 1–10 olsun

**Sorun:** hücrelerin %22–52'si aynı taban değerinde (etrafında hiç POI yok).
`pd.qcut(..., duplicates="drop")` eşit kesim noktalarını siliyor → 10 yerine 5 bant.

**Çözüm:** tabandaki kütleyi *ayrı tut*, doğrudan 1. banda ver. Kalan hücreleri
9 eşit dilime böl. Böylece 10 bandın hepsi dolu ve tavan gerçekten 10.

In [ ]:
def fit_cutpoints(series, n=10):
    """Bant sınırlarını öğren. Beraberliklere dayanıklı."""
    s = pd.Series(series).astype(float)
    floor = s.min()
    rest = s[s > floor]                    # -> tabanda olmayanlar
    if rest.empty:
        return {"floor": float(floor), "edges": []}
    # -> Kalanları 9 dilime böl (1. bant zaten tabana ayrıldı)
    qs = np.linspace(0, 1, n)[1:-1]
    edges = np.unique(np.quantile(rest, qs))
    return {"floor": float(floor), "edges": [float(e) for e in edges]}

def apply_cutpoints(series, cuts):
    """Öğrenilmiş sınırlarla puanla. Yeni veriyi de bu sınırlarla puanlayabiliriz."""
    s = pd.Series(series).astype(float)
    out = np.searchsorted(np.array(cuts["edges"]), s.values, side="right") + 2
    out = np.where(s.values <= cuts["floor"], 1, out)   # -> taban = 1
    return np.clip(out, 1, 10).astype(np.int8)

## Düzeltme 5 · Mesafeleri sınırla

"En yakın metro 39 km" bir erişilebilirlik bilgisi değil. Kategori başına
gerçekçi bir hizmet yarıçapı; ötesi `NULL` → platform "—" gösterir, uydurma sayı değil.

In [ ]:
SERVICE_RADIUS_M = {"kdgr": 2000, "schl": 2000, "metro": 3000, "shopping": 5000, "parks": 2000}
NEAREST = {"kindergarten": "kdgr", "schl": "schl", "metro": "metro",
           "shopping_mall": "shopping", "parks": "parks"}

def nearest(cells, poi, cap=None):
    tree = BallTree(np.radians(poi[["lat", "lon"]].values), metric="haversine")
    d, i = tree.query(np.radians(cells[["latitude", "longitude"]].values), k=1)
    d = d.ravel() * R_EARTH
    name = poi["name"].values[i.ravel()].astype(object)
    if cap is not None:
        beyond = d > cap          # -> hizmet yarıçapının ötesi anlamsız
        d = np.where(beyond, np.nan, d)
        name = np.where(beyond, None, name)
    return d, name

## İndeks grupları

Orijinaldeki altı grup + **yedincisi**: `safety`, 2026'da gelen kamera ve
yanğından mühafizə verisinden.

In [ ]:
GROUPS = {
    "education":  ["kdgr", "schl", "univ"],
    "transport":  ["metro", "bus"],
    "tourism":    ["museum", "theatre", "hotel", "monuments", "concert_hall"],
    "food":       ["coffee", "restaurant", "fast_food"],          # -> bar hariç (orijinal karar)
    "recreation": ["parks", "sport"],
    # -> leisure, food ve recreation üyelerini tekrar kullanıyor -> indeksler korele (B4)
    "leisure":    ["museum", "shopping", "parks", "theatre", "concert_hall",
                   "coffee", "restaurant", "fast_food", "bars"],
    "safety":     ["camera", "fire"],     # -> YENİ, sadece 2026 verisiyle mümkün
}

def build(which):
    """Bir veri seti için tüm skorları ve indeksleri üret."""
    t0 = time.time()
    pois = load_pois(which)
    df = cells.copy()
    # -> Önce kategori skorları (sürekli azalım)
    sc = {f"{k}_score": continuous_score(cells, v, DECAY[k]) for k, v in pois.items()}
    df = pd.concat([df, pd.DataFrame(sc, index=df.index)], axis=1)

    # -> Sonra grup indeksleri (PCA, işaret sabitli)
    meta, idx = {"pca": {}}, {}
    for g, members in GROUPS.items():
        members = [m for m in members if f"{m}_score" in df.columns]
        if not members: continue
        comp, load, evr = pca_index(df[[f"{m}_score" for m in members]])
        idx[f"{g}_index"] = comp
        meta["pca"][g] = {"explained_variance": float(evr),
                          "loadings": {m: float(v) for m, v in zip(members, load)}}
    df = pd.concat([df, pd.DataFrame(idx, index=df.index)], axis=1)

    # -> En yakın mesafeler (sınırlı)
    nn = {}
    for pretty, label in NEAREST.items():
        d, nm = nearest(cells, pois[label], SERVICE_RADIUS_M.get(label))
        nn[f"distance_to_nearest_{pretty}"] = d
        nn[f"nearest_{pretty}_name"] = nm
    df = pd.concat([df, pd.DataFrame(nn, index=df.index)], axis=1)
    print(f"  {which}: {time.time()-t0:.0f} sn")
    return df, meta

df_new, meta_new = build("new")
df_old, meta_old = build("old")

print("\nPCA sonuçları (2026):")
for g, m in meta_new["pca"].items():
    print(f"  {g:<11} açıklanan varyans={m['explained_variance']:6.1%}   "
          + "  ".join(f"{k}={v:+.2f}" for k, v in m["loadings"].items()))

## Düzeltme 1'in sonucu — artık tavan gerçekten 10

In [ ]:
G = [g for g in GROUPS if f"{g}_index" in df_new.columns]
print(f"{'indeks':<12}{'bant':>6}{'tavan':>7}   bant başına hücre")
for g in G:
    cuts = fit_cutpoints(df_new[f"{g}_index"])
    df_new[f"{g}_score_1_10"] = apply_cutpoints(df_new[f"{g}_index"], cuts)
    vc = df_new[f"{g}_score_1_10"].value_counts().sort_index()
    print(f"{g:<12}{len(vc):>6}{int(vc.index.max()):>7}   {[int(x) for x in vc.values]}")

print("\nKarşılaştırma — orijinal yöntem aynı veride ne veriyordu:")
for g in G:
    if f"{g}_index" not in df_new.columns: continue
    d = pd.qcut(df_new[f"{g}_index"], 10, labels=False, duplicates="drop") + 1
    print(f"  {g:<12} orijinal tavan={int(d.max()):2d}/10   ->   yeni tavan=10/10")

## Düzeltme 2 · İndeksi *dondurulmuş model* haline getir

Bu, raporda **B2**. En önemli düzeltme ve anlaşılması en zor olanı, o yüzden açayım.

Orijinal kod her çalıştığında şunları **sıfırdan** öğreniyor:
- `StandardScaler` ortalamaları ve standart sapmaları
- PCA yükleri
- Desil sınırları

Yani referans çerçevesinin **kendisi** kayıyor. Bir hücrenin çevresi hiç değişmese
bile puanı oynuyor — çünkü *başka* hücreler değişti.

**Çözüm:** 2025'i bir kez *fit* et, sonucu diske yaz, 2026'yı o modelle *transform* et.

In [ ]:
def fit_model(df, groups):
    """2025 verisiyle bir kez öğren: ölçekleyici + PCA + bant sınırları."""
    model = {"groups": {}}
    for g, members in groups.items():
        members = [m for m in members if f"{m}_score" in df.columns]
        if not members: continue
        X = df[[f"{m}_score" for m in members]].values
        sc = StandardScaler().fit(X)                  # -> ortalama ve std'yi SAKLA
        p = PCA(n_components=1).fit(sc.transform(X))  # -> yükleri SAKLA
        comp = p.components_[0]
        flip = -1.0 if comp.sum() < 0 else 1.0        # -> işaret sabitlemesi
        idx = p.transform(sc.transform(X)).ravel() * flip
        model["groups"][g] = {
            "members": members,
            "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
            "components": (comp * flip).tolist(),
            "explained_variance_ratio": float(p.explained_variance_ratio_[0]),
            "cutpoints": fit_cutpoints(idx),          # -> bant sınırlarını da SAKLA
        }
    return model

def score(df, model):
    """Kaydedilmiş modelle puanla. fit_transform DEĞİL — sadece transform."""
    out = {}
    for g, m in model["groups"].items():
        X = df[[f"{x}_score" for x in m["members"]]].values
        Z = (X - np.array(m["mean"])) / np.array(m["scale"])   # -> SAKLANMIŞ ortalama/std
        idx = Z @ np.array(m["components"])                    # -> SAKLANMIŞ yükler
        out[f"{g}_index"] = idx
        out[f"{g}_score_1_10"] = apply_cutpoints(idx, m["cutpoints"])  # -> SAKLANMIŞ sınırlar
    return pd.DataFrame(out, index=df.index)

# -> 2025 = temel yıl. Modeli diske yaz ki gelecek yıl da aynısı kullanılsın.
model = fit_model(df_old, GROUPS)
with open(str(OUT / "frozen_model_2025.json"), "w", encoding="utf-8") as f:
    json.dump(model, f, indent=2)
print("model yazıldı:", str(OUT / "frozen_model_2025.json"))
print("saklanan gruplar:", list(model["groups"]))

### Fark ne kadar? İki yöntemi yan yana koyalım

In [ ]:
s_old, s_new = score(df_old, model), score(df_new, model)
key = df_old[["h3_index"]].join(s_old).merge(
      df_new[["h3_index"]].join(s_new), on="h3_index", suffixes=("_o", "_n"))
GG = list(model["groups"])

print("A) ORİJİNAL YOL — her seferinde sıfırdan fit (referans çerçevesi kayıyor)")
for g in GG:
    a = pd.qcut(df_old[f"{g}_index"], 10, labels=False, duplicates="drop")
    b = pd.qcut(df_new[f"{g}_index"], 10, labels=False, duplicates="drop")
    m = df_old[["h3_index"]].assign(a=a).merge(df_new[["h3_index"]].assign(b=b), on="h3_index")
    print(f"   {g:<12} oynayan hücre: {(m.a != m.b).mean()*100:5.1f}%")

print("\nB) DONDURULMUŞ MODEL — sadece kendi çevresi değişen hücre oynuyor")
print(f"   {'indeks':<12}{'oynayan':>9}{'ort. Δ':>9}{'yükselen':>10}{'düşen':>8}")
for g in GG:
    a = key[f"{g}_score_1_10_o"].astype(int); b = key[f"{g}_score_1_10_n"].astype(int)
    d = b - a
    print(f"   {g:<12}{(d!=0).mean()*100:8.1f}%{d.mean():+9.3f}{(d>0).mean()*100:9.1f}%{(d<0).mean()*100:7.1f}%")
print("\n-> B'de hareket hem çok daha az, hem de neredeyse tamamı YUKARI yönlü.")
print("   Mantıklı: 2026'da her kategoride yeni POI açılmış, kapanan neredeyse yok.")

## Bakı'da gerçekte ne değişti?

Dondurulmuş modelle rayon rayon bakalım — bu, ekibin asıl merak ettiği soru.

In [ ]:
# -> Her hücreyi rayonuna eşle
RAYON = str(REFDIR / "baku_rayons.geojson")
import shapely
b = gpd.read_file(RAYON)
# -> Bu dosyada iki kusur var: (1) Gəncə'nin Nizami rayonu yanlışlıkla duruyor (boylam ~46.3),
# -> (2) altı poligon kendini kesiyor -> GEOS hata veriyor. İkisini de onarıyoruz.
b = b[b.geometry.bounds["minx"] > 48.5].copy()
b["geometry"] = shapely.make_valid(b.geometry.values)

pts = gpd.GeoDataFrame(
    key.merge(df_new[["h3_index","latitude","longitude"]], on="h3_index"),
    geometry=gpd.points_from_xy(
        key.merge(df_new[["h3_index","latitude","longitude"]], on="h3_index").longitude,
        key.merge(df_new[["h3_index","latitude","longitude"]], on="h3_index").latitude),
    crs=4326)
j = gpd.sjoin(pts, b[["name","geometry"]], how="left", predicate="within")

rows = []
for g in GG:
    d = j[f"{g}_score_1_10_n"].astype(int) - j[f"{g}_score_1_10_o"].astype(int)
    rows.append(j.assign(d=d).groupby("name")["d"].mean().rename(g))
tab = pd.concat(rows, axis=1).round(2)
tab["hücre"] = j.groupby("name").size()
print("Rayon başına ORTALAMA bant değişimi (2025 -> 2026):")
tab.sort_values("transport", ascending=False)

**Okunuşu:**

- **Pirallahı ve Xəzər turizmde önde** — Abşeron sahil şeridinde yeni oteller
- **Suraxanı ve Sabunçu nəqliyyatda** — 219 yeni dayanacaq ağırlıklı dış rayonlara gitmiş
- **Nəsimi ve Yasamal'da sıfır** — zaten tavandalar, yeni POI puanı artıramıyor (tavan etkisi)

Yani: **merkez doymuş, çeper büyüyor.**

## Yeni yedinci indeks: təhlükəsizlik

In [ ]:
saf = df_new["safety_index"]
print(f"açıklanan varyans: {meta_new['pca']['safety']['explained_variance']:.1%}")
print(f"yükler: {meta_new['pca']['safety']['loadings']}")
print("\nDiğer indekslerle korelasyonu:")
for g in ["education","transport","tourism","food","recreation","leisure"]:
    print(f"  {g:<12}{saf.corr(df_new[f'{g}_index']):.3f}")
print("\n-> Korele ama fazlalık değil; mevcut altısı da birbiriyle 0.58-0.93 arası korele.")

## Bonus bulgu · Altı indeks aslında tek bir şey mi ölçüyor?

İndeksleri **birbirine karşı** analiz edelim.

In [ ]:
X = StandardScaler().fit_transform(df_new[[f"{g}_index" for g in GG]])
p = PCA().fit(X)
for i, (evr, cum) in enumerate(zip(p.explained_variance_ratio_, np.cumsum(p.explained_variance_ratio_)), 1):
    print(f"  PC{i}: {evr:6.1%}   kümülatif {cum:6.1%}")
print(f"\n-> Tüm farklılığın {p.explained_variance_ratio_[0]:.0%}'i TEK bir faktör: 'merkeze ne kadar yakınsın'.")
print("   PC1 yükleri (hepsi neredeyse eşit -> hepsi aynı şeyi söylüyor):")
for g, l in zip(GG, p.components_[0]): print(f"     {g:<12}{l:+.3f}")
print("\n   PC2 — indeksleri asıl AYIRAN eksen:")
for g, l in zip(GG, p.components_[1]): print(f"     {g:<12}{l:+.3f}")
print("\n-> Müşteriye altı ayrı bal gösteriyoruz ama %84'ü aynı sayı. Ürün kararı: S9.")

## Çıktı — geliştiricilere gidecek dosya

In [ ]:
cols = (["h3_index", "latitude", "longitude"]
        + [c for c in df_new.columns if c.startswith("distance_to_nearest_")]
        + [f"{g}_index" for g in GG] + [f"{g}_score_1_10" for g in GG])
out = df_new[cols].copy()
for g in GG: out[f"{g}_index"] = out[f"{g}_index"].round(4)
for c in cols:
    if c.startswith("distance_to_nearest_"): out[c] = out[c].round(2)

path = str(OUT / "poi_scores_2026_v2.csv")
out.to_csv(path, index=False)
print(f"yazıldı: {path}\n{len(out):,} satır × {len(out.columns)} sütun "
      f"({os.path.getsize(path)/1e6:.1f} MB)")
out[[f"{g}_score_1_10" for g in GG]].describe().round(2)

---

## Özet

Yedi indeks, hepsi 1–10 arası ve tavanı gerçekten 10. Model diske yazıldı
(`out_v2/frozen_model_2025.json`), böylece gelecek yılın verisi aynı ölçekle
puanlanabilir.

**Ama üretime vermeden önce dört cevap lazım** — `suallar.html`'deki S1–S4:

1. `map_absheron.geojson` nerede? (buradaki ızgara GADM ikamesiyle üretildi)
2. Metro verisi dosyadan mı gelecek, OSM'den mi?
3. `konsert zali.csv` neden yenilenmemiş?
4. Platform hangi H3 çözünürlüğünde birleştiriyor? (bu çıktı res-10, platform res-6)